# Entraînement du classifieur d'espèces — Fisher Link

Ce notebook entraîne le modèle utilisé par `server/fishid.js` (voir `ml/README.md`), sur Google Colab avec un GPU gratuit.

**Avant de commencer :** Menu **Exécution → Modifier le type d'exécution → GPU (T4)**.

Jeu de données : [A Large Scale Fish Dataset](https://www.kaggle.com/datasets/crowww/a-large-scale-fish-dataset) (9 espèces commerciales — bar, dorade, mulet, maquereau, truite, etc.). Ce n'est qu'un premier modèle de départ : affinez-le plus tard avec vos propres photos de prises réelles (voir `ml/README.md` § Réentraînement).

In [ ]:
import torch
print("GPU disponible :", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("⚠ Pas de GPU détecté — vérifiez Exécution > Modifier le type d'exécution > GPU")

## 1. Récupérer le jeu de données depuis Kaggle

Il faut une clé API Kaggle (gratuite) :
1. Créez un compte sur [kaggle.com](https://www.kaggle.com) si besoin.
2. Allez dans **Paramètres du compte → API → Create New Token**. Cela télécharge un fichier `kaggle.json`.
3. Exécutez la cellule suivante, elle vous demandera d'uploader ce fichier.

In [ ]:
from google.colab import files
uploaded = files.upload()  # sélectionnez kaggle.json

import os
os.makedirs("/root/.kaggle", exist_ok=True)
!cp kaggle.json /root/.kaggle/kaggle.json
!chmod 600 /root/.kaggle/kaggle.json

In [ ]:
!pip install -q kaggle
!kaggle datasets download -d crowww/a-large-scale-fish-dataset -p /content/raw
!unzip -q -o /content/raw/a-large-scale-fish-dataset.zip -d /content/raw/fish

## 2. Vérifier la structure du jeu de données

Cette cellule affiche l'arborescence (2 niveaux) pour qu'on voie exactement comment c'est organisé avant de préparer les données — évite de deviner des noms de dossiers qui pourraient être inexacts.

In [ ]:
import pathlib
root = pathlib.Path("/content/raw/fish")
for p in sorted(root.rglob("*")):
    if p.is_dir():
        depth = len(p.relative_to(root).parts)
        if depth <= 3:
            print("  " * depth + p.name)

## 3. Préparer les données au format attendu par `train.py`

`train.py` attend `data/<espèce>/*.jpg`. Ce jeu de données Kaggle contient, pour chaque espèce, un dossier d'images brutes **et** un dossier de masques "GT" (ground truth, à ignorer — ce ne sont pas des photos). La cellule ci-dessous parcourt l'arborescence automatiquement et ne garde que les dossiers-feuilles qui ne se terminent PAS par "GT", regroupés par le nom du dossier parent (l'espèce) — ça évite de dépendre de noms exacts de sous-dossiers qu'on n'a pas pu vérifier à l'avance.

In [ ]:
import shutil

data_dir = pathlib.Path("/content/data")
shutil.rmtree(data_dir, ignore_errors=True)
data_dir.mkdir()

img_exts = {".jpg", ".jpeg", ".png"}
n_copied = 0
for d in root.rglob("*"):
    if not d.is_dir() or d.name.strip().lower().endswith("gt"):
        continue
    images = [f for f in d.iterdir() if f.suffix.lower() in img_exts]
    if not images:
        continue
    species = d.parent.name if d.parent != root else d.name
    species = species.strip()
    dest = data_dir / species
    dest.mkdir(parents=True, exist_ok=True)
    for f in images:
        shutil.copy(f, dest / f.name)
        n_copied += 1

print(f"{n_copied} images copiées dans {data_dir}")
for d in sorted(data_dir.iterdir()):
    print(" -", d.name, ":", len(list(d.iterdir())), "images")

**Vérifiez la sortie ci-dessus** : vous devriez voir 9 espèces avec un nombre d'images cohérent (des centaines chacune). Si un dossier a 0 ou trop peu d'images, ou si les noms d'espèces semblent faux (ex: contiennent encore "GT"), corrigez le filtre de la cellule précédente avant de continuer.

## 4. Récupérer les scripts d'entraînement (`ml/train.py`, `ml/export_onnx.py`)

On clone le dépôt GitHub du projet pour réutiliser exactement les mêmes scripts que ceux du dossier `ml/` local — pas de duplication de code.

In [ ]:
!rm -rf /content/repo
!git clone --depth 1 https://github.com/KingjulianB/ha-addon-peche.git /content/repo
%cd /content/repo/peche/ml
!pip install -q onnx

## 5. Entraîner

`torch`/`torchvision` sont déjà installés sur Colab. 10 époques est un point de départ raisonnable pour un premier modèle — augmentez si la précision de validation continue de progresser.

In [ ]:
!python train.py --data-dir /content/data --epochs 10 --out /content/model.pt --labels /content/labels.json

## 6. Exporter au format ONNX

In [ ]:
!python export_onnx.py --model /content/model.pt --labels /content/labels.json --out /content/model.onnx

## 7. Télécharger le modèle

Placez ensuite les deux fichiers téléchargés dans `ha-addon/peche/ml/` en local (à côté de `species_weights.json`), redémarrez le serveur : `fishIdConfigured()` deviendra automatiquement vrai.

In [ ]:
from google.colab import files
files.download("/content/model.onnx")
files.download("/content/labels.json")